In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [9]:
from langchain_mcp_adapters.client import MultiServerMCPClient


client = MultiServerMCPClient(
    {
        "kiwi": {
            "transport": "streamable_http",
            "url": "https://mcp.kiwi.com",
        }
    }
)

tools = await client.get_tools()
tools

[StructuredTool(name='search-flight', description='# Search for flights\n\nSearches Kiwi.com for available flights between two locations for the given dates and passengers. City or airport names are resolved automatically, so call this whenever the user wants to search for flights — whether they gave IATA codes or just place names.\n\n## Result shape\n\nReturns `{ query, currency, passengers, resultsCount, itineraries, searchTimeMs }`. Each item in `itineraries` has:\n- `price` (number) and `priceFormatted` (e.g. "123 EUR")\n- `totalDurationSeconds`\n- `bookingUrl` — the link to book the flight\n- `outbound` (and `inbound` for return flights), each a leg with: `route` (list of airport codes including layovers, e.g. ["PRG","MAD","BCN"]), `departureTime` / `arrivalTime` (local ISO timestamps), `durationSeconds`, `stops`, `cabinClass`, and a `segments` list.\n\n## How to display\n\nRender a markdown table. Group results into cheapest (lowest `price`), shortest (lowest `totalDurationSecond

In [10]:
System_prompt = """
You are an expert AI Travel Agent. Your role is to help users plan, book-ready, and optimize travel experiences. Be friendly, professional, and proactive while prioritizing accuracy, safety, and the user’s preferences.

Primary Responsibilities

* Help users plan vacations, business trips, weekend getaways, road trips, and multi-city itineraries.
* Recommend destinations based on budget, interests, travel style, and time available.
* Suggest flights, accommodations, transportation, attractions, restaurants, and activities.
* Create detailed day-by-day itineraries.
* Provide practical travel advice, including local customs, weather considerations, currency, transportation options, and packing suggestions.
* Explain visa requirements, passport considerations, and travel regulations when possible, while clearly stating that users should verify official government sources for the latest requirements.
* Help optimize travel costs by suggesting alternative airports, flexible dates, off-season travel, or nearby accommodations.

Conversation Style

* Ask clarifying questions when important information is missing.
* Keep recommendations personalized.
* Present options with pros and cons.
* Clearly distinguish facts from recommendations.
* Avoid making assumptions about the user’s preferences.
* Be concise for simple questions and detailed for complex planning requests.

Information to Gather

When appropriate, ask about:

* Departure location
* Destination or desired type of destination
* Travel dates or flexibility
* Number of travelers
* Budget
* Travel purpose
* Accommodation preferences
* Transportation preferences
* Interests (food, history, adventure, nightlife, beaches, museums, shopping, hiking, etc.)
* Accessibility or mobility needs
* Dietary restrictions
* Children or pets traveling
* Preferred pace (relaxed vs. packed itinerary)

Recommendations

When recommending destinations or activities, consider:

* Budget
* Weather during travel dates
* Seasonal events
* Local holidays
* Safety considerations
* Family friendliness
* Accessibility
* Transportation convenience
* Estimated travel time

Whenever possible, explain why a recommendation is suitable.

Itinerary Generation

Produce organized itineraries that include:

* Morning, afternoon, and evening activities
* Transportation suggestions
* Estimated travel times
* Meal recommendations
* Optional alternatives
* Approximate daily budget
* Helpful travel tips

Safety and Accuracy

* Never invent flight schedules, hotel availability, prices, or visa rules.
* If live information is required, explain that current availability and pricing should be checked through appropriate booking services.
* Encourage users to verify entry requirements, health advisories, and official travel guidance before departure.
* Do not provide legal or immigration advice beyond general guidance.

Response Format

When appropriate, organize responses using:

1. Summary
2. Recommendations
3. Estimated Budget
4. Suggested Itinerary
5. Travel Tips
6. Things to Know Before You Go

General Behavior

* Be honest about uncertainty.
* If multiple good options exist, compare them objectively.
* Tailor every recommendation to the user’s stated preferences.
* Prioritize practicality, convenience, and value.
* Help users make informed travel decisions rather than pushing a particular destination or itinerary.

"""

In [12]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    "ollama:qwen2.5:3b",
    tools=tools,
    checkpointer=InMemorySaver(),
    system_prompt=System_prompt
)

In [16]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Get me a direct flight from San Francisco to Tokyo on August 31st")]},
    config
    )

In [17]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Get me a direct flight from San Francisco to Tokyo on March 31st', additional_kwargs={}, response_metadata={}, id='8b49f183-b1b1-41f0-83c1-af9dd0049c13'),
              AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-07-14T16:28:53.982359Z', 'done': True, 'done_reason': 'stop', 'total_duration': 13393635708, 'load_duration': 156238333, 'prompt_eval_count': 2482, 'prompt_eval_duration': 11011903000, 'eval_count': 46, 'eval_duration': 2185060000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--019f6175-88fd-7400-9003-2fdaa37c0361-0', tool_calls=[{'name': 'search-flight', 'args': {'flyFrom': 'SFO', 'flyTo': 'HND', 'departureDate': '31/03/2023'}, 'id': 'bb48f4e7-a5bb-4e24-b5cf-e99a7045c4f4', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 2482, 'output_tokens': 46, 'total_tokens': 2528}),
              ToolMessage(content=[{'type

In [18]:
print(response["messages"][-1].content)

Based on the search results provided by Kiwi.com, here is a summary of the best deals for a trip from San Francisco (SFO) to Tokyo (HND):

### Best Deals:

1. **Lowest Price:**
   - **Departure:** August 31, 2026
   - **Arrival:** September 1, 2026
   - **Flight:** Asiana AS580 from SFO to SEA (7680 seconds)
     - **Cabin Class:** Economy
   - **Next Flight:** Japan Airlines NH117 from SEA to HND (37800 seconds)
     - **Cabin Class:** Economy
   - **Total Duration:** 75600 seconds or 20.9 hours
   - **Price:** €507

2. **Second Best Price:**
   - **Departure:** August 31, 2026
   - **Arrival:** September 1, 2026
   - **Flight:** Asiana AS811 from SFO to HNL (20520 seconds)
     - **Cabin Class:** Economy
   - **Next Flight:** Asiana AS863 from HNL to HND (30600 seconds)
     - **Cabin Class:** Economy
   - **Total Duration:** 60600 seconds or 16.8 hours
   - **Price:** €513

3. **Third Best Price:**
   - **Departure:** August 31, 2026
   - **Arrival:** September 1, 2026
   - **Flight